# Compare a noise budget with a sampled trace
How much noise comes from shot noise, thermal noise and the ADC? Use a weak reflector to drive the ADC across many codes. Subtract a reference trace with the same seed and receiver settings, then compare the measured standard deviations with the current noise predicted by `compute_budget`.

In [1]:
from copy import deepcopy
import numpy as np

from pyofdr.core.config import load_config
from pyofdr.core.campaign import run_campaign
from pyofdr.analysis.budget import compute_budget

cfg = load_config('../configs/ofdr_basic.yaml')
cfg['source'].update(sweep_duration=0.002, linewidth=0, rin_dB_per_Hz=None)
cfg['fiber'].update(length=0.5, rayleigh_coefficient_dB=-300,
                    reflectors=[{'z': 0.3, 'R': 1e-4}])
cfg['adc']['sample_rate'] = 100e6
cfg['detection'].update(bandwidth=10e6, shot_noise=False, thermal_nep=0, dark_current=0)
reference = run_campaign(cfg)[0]
baseline = reference.analog_main[0]
Z = cfg['adc']['input_impedance']
lsb = cfg['adc']['voltage_range'] / 2 ** cfg['adc']['bits']
middle = slice(len(baseline) // 10, -len(baseline) // 10)

## Receiver and ADC contributions
Shot and thermal noise pass through the receiver filter. Quantization happens after filtering and has a nominal standard deviation of one LSB divided by $\sqrt{12}$ in volts. Report each standard deviation in nA referred to the receiver input. For the combined row compare reconstructed ADC voltage with the clean **analog** trace, so the subtraction contains one quantization error. Using the standard deviation removes the floor quantizer's mean offset.

In [2]:
cases = [
    ('shot', {'shot_noise': True}, 'sigma_shot'),
    ('thermal', {'thermal_nep': 1e-9}, 'sigma_thermal'),
    ('quantization', {}, 'sigma_quant'),
    ('combined', {'shot_noise': True, 'thermal_nep': 1e-9}, 'sigma_total'),
]
print(f"{'source':<15} {'budget [nA]':>12} {'measured [nA]':>14} {'ratio':>8}")
for name, receiver, key in cases:
    trial = deepcopy(cfg)
    trial['detection'].update(receiver)
    acq = run_campaign(trial)[0] if receiver else reference
    if name in ('quantization', 'combined'):
        signal = acq.digital_main[0].astype(float) * lsb
    else:
        signal = acq.analog_main[0]
    measured = np.std((signal[middle] - baseline[middle]) / Z)
    predicted = compute_budget(trial)[key]
    print(f"{name:<15} {predicted * 1e9:12.2f} {measured * 1e9:14.2f} {measured / predicted:8.3f}")

source           budget [nA]  measured [nA]    ratio
shot                   90.35          90.55    1.002
thermal              3192.26        3204.25    1.004
quantization          176.19         178.22    1.012
combined             3198.40        3210.22    1.004


The combined prediction adds the independent variances. A ratio close to one indicates agreement for this bandwidth, input amplitude and ADC resolution. Try changing `bits` or the receiver bandwidth and rerun from the reference cell. The uniform quantization estimate relies on a signal that spans multiple codes without clipping.